### Purpose

거래처 부분 수정 · 별칭 · 변경 이력 API 배포 (docs/gdb-update-api-request.md, 재고 관리 프로젝트 요청).

| 대상 | 내용 |
|---|---|
| DynamoDB `dscustomer_history` | 키 `custcd` + `at` — 변경 이력 |
| Lambda `dsglobaldbCreate` (갱신) | `POST /dscustomer/update`, `POST /dscustomer/alias`, `GET /dscustomer/history` |

- 기존 `/dscustomer` GET/POST/PUT/DELETE 는 건드리지 않는다
- 인증 없음 (기존 API 와 같음), 패스스루 매핑 템플릿, CORS `*`, 역할 `DSmapLambda`
- 배포 전 테스트: `pytest AWS/Lambda/dsglobaldbCreate`
- **마지막 셀(`dev` 스테이지 배포)은 API 전체를 다시 배포한다**

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, get_request_template, setup_lambda_method,
                            setup_options, grant_invoke, deploy_stage_and_check)

Settings

In [ ]:
HISTORY_TABLE = 'dscustomer_history'
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'   # dynamodb:* on table/* 권한 있음

API_ID = 'jyipsj28s9'
STAGE = 'dev'
TEMPLATE_FROM = '/dschemical'

FUNCTION_NAME = 'dsglobaldbCreate'
SOURCE_FILE = './dsglobaldbCreate/lambda_function.py'
ROUTES = {
    'dscustomer/update': ['POST'],
    'dscustomer/alias': ['POST'],
    'dscustomer/history': ['GET'],
}
ACCOUNT_ID = ROLE_ARN.split(':')[4]

1. 변경 이력 테이블 (on-demand)

In [ ]:
ddb = boto3.client('dynamodb')
if HISTORY_TABLE in ddb.list_tables()['TableNames']:
    print('테이블 있음:', HISTORY_TABLE)
else:
    ddb.create_table(
        TableName=HISTORY_TABLE,
        KeySchema=[{'AttributeName': 'custcd', 'KeyType': 'HASH'}, {'AttributeName': 'at', 'KeyType': 'RANGE'}],
        AttributeDefinitions=[{'AttributeName': 'custcd', 'AttributeType': 'S'}, {'AttributeName': 'at', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST',
    )
    ddb.get_waiter('table_exists').wait(TableName=HISTORY_TABLE)
    print('테이블 생성:', HISTORY_TABLE)

2. Lambda 갱신 + API Gateway 경로 + 권한

In [ ]:
function_arn = deploy_lambda(FUNCTION_NAME, SOURCE_FILE, ROLE_ARN,
                             description='글로벌 DB 안전한 쓰기 (create · 거래처 update/alias/history)', timeout=29, memory=256)
request_template = get_request_template(API_ID, TEMPLATE_FROM)
for path, methods in ROUTES.items():
    resource_id = ensure_resource_path(API_ID, path)
    for m in methods:
        setup_lambda_method(API_ID, resource_id, m, function_arn, request_template)
    setup_options(API_ID, resource_id, methods)
    grant_invoke(FUNCTION_NAME, API_ID, ACCOUNT_ID, path)

3. `dev` 스테이지 배포 + 확인 (이력 조회, 쓰기 없음)

In [ ]:
res_ = deploy_stage_and_check(API_ID, STAGE, 'dscustomer/history?id=__check__',
                              'add /dscustomer/update, /dscustomer/alias, /dscustomer/history')
print('statusCode', res_['statusCode'], json.loads(res_['body']))